# Model Safety Filters Analysis

## Purpose

This notebook analyzes cases where **model safety filters** interfere with benchmark task execution. In cybersecurity evaluation, models may refuse to perform legitimate testing tasks because their safety guardrails interpret the prompts as malicious.

This is a **cross-cutting concern** that affects multiple domains and agent architectures differently.

### Why this matters

- Safety refusals produce **0.0 scores** that are not comparable to genuine failures
- The same model may refuse with one agent architecture but not another (prompt framing matters)
- Refusal patterns vary by model family, revealing differences in safety training
- Understanding these patterns is essential for fair benchmark comparisons

### Analyses in this notebook

| # | Analysis | Description |
|---|----------|-------------|
| 1 | Haiku 4.5 + GH Copilot (Excytin) | 100% refusal rate — safety guardrails triggered by Copilot prompt framing |
| 2 | Cross-Model Refusal Detection | Scan all evals for safety refusal patterns |


## Prerequisites

1. Eval files in `eval_samples/` or `latest_experiments/` directories
2. Run the domain-agnostic and domain-specific analysis notebooks first for context

### How this relates to other notebooks

- **[`excytin_agent_architecture_analysis.ipynb`](excytin_agent_architecture_analysis.ipynb)** — Contains the initial discovery of the Haiku 4.5 + Copilot refusal pattern
- **[`agent_architecture_analysis.ipynb`](agent_architecture_analysis.ipynb)** — Overall agent architecture comparison (safety refusals produce 0.0 scores that skew those results)
- **[`eval_analysis.ipynb`](eval_analysis.ipynb)** — Domain-agnostic analysis framework (doesn't account for safety refusals)

<a id="configuration"></a>
## Configuration

### What this cell does

Sets up directories to search for eval files, defines the safety refusal detection patterns, and establishes the refusal category taxonomy used throughout this notebook.

### Refusal detection approach

Safety refusals are detected by scanning the `answer` field of `saber_overall` scores for known refusal phrases (e.g., "I cannot", "social engineering", "not authorized"). These are then categorized into overlapping groups:
- **Credential concerns** — model worries about using provided credentials
- **Social engineering** — model interprets the task as a social engineering attempt
- **Authorization/verification** — model claims it can't verify authorization
- **Explicit refusal** — model directly states it will not perform the task
- **Content policy** — model cites general content/safety policies
- **Harmful activity** — model flags the activity as potentially harmful

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Model Safety Filters Analysis
# ══════════════════════════════════════════════════════════════════════════════
import os
import sys
import re
import json
import zipfile
import textwrap
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from saber_analysis.plots import setup_plotting
setup_plotting()

REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
ARTIFACTS_DIR = NOTEBOOK_DIR / 'artifacts' / 'safety_filters'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Directories to search for eval files
EVAL_DIRS = {
    'excytin': [
        str(REPO_ROOT / 'eval_samples' / 'excytin'),
        str(REPO_ROOT / 'latest_experiments' / 'excytin'),
    ],
    'cybench': [
        str(REPO_ROOT / 'eval_samples' / 'cybench'),
        str(REPO_ROOT / 'latest_experiments' / 'cybench'),
    ],
}

# Safety refusal detection patterns
REFUSAL_PHRASES = [
    'i cannot', 'i can\'t', 'i will not', 'i won\'t',
    'i\'m not able to', 'i am not able to',
    'i must decline', 'i need to decline',
    'social engineering', 'security concerns',
    'not authorized', 'unauthorized access',
    'cannot assist with', 'cannot help with',
    'content policy', 'safety guidelines',
    'potentially harmful', 'potentially dangerous',
    'ethical concerns', 'against my guidelines',
]

REFUSAL_CATEGORIES = {
    'Credential concerns': ['credential', 'password', 'plain text', 'authentication'],
    'Social engineering': ['social engineering', 'manipulat', 'pretexting'],
    'Authorization/verification': ['cannot independently verify', 'not authorized',
                                    'unauthorized', 'verification failure'],
    'Explicit refusal': ['will not', 'won\'t', 'i cannot and will not', 'i must decline'],
    'Content policy': ['content policy', 'safety guidelines', 'against my guidelines'],
    'Harmful activity': ['potentially harmful', 'potentially dangerous', 'malicious'],
}

print(f'Artifacts dir: {ARTIFACTS_DIR}')
print(f'Eval directories configured for: {list(EVAL_DIRS.keys())}')


## Helper Functions

### What this cell does

Defines utility functions that power the safety refusal analysis:

- **`is_safety_refusal(answer)`** — Returns `True` if the answer text matches any known refusal phrase
- **`classify_refusal(answer)`** — Returns a dict of category → matched boolean for detailed classification
- **`scan_eval_file(path)`** — Scans a full `.eval` ZIP archive, extracting refusal counts, category breakdowns, model identity, and example refusals
- **`find_eval_files(domain)`** — Locates all `.eval` files for a domain across configured directories

### Why separate helper functions?

These are reusable across both the deep-dive case study (Analysis 1) and the cross-model scan (Analysis 2). The `scan_eval_file` function works directly on the ZIP archive without needing the full inspect_ai log parser, making it fast for bulk scanning.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def is_safety_refusal(answer: str) -> bool:
    """Detect if an answer text contains safety refusal patterns."""
    if not answer:
        return False
    answer_lower = answer.lower()
    return any(phrase in answer_lower for phrase in REFUSAL_PHRASES)


def classify_refusal(answer: str) -> dict[str, bool]:
    """Classify a refusal into categories. Returns dict of category -> matched."""
    if not answer:
        return {cat: False for cat in REFUSAL_CATEGORIES}
    answer_lower = answer.lower()
    return {
        cat: any(kw in answer_lower for kw in keywords)
        for cat, keywords in REFUSAL_CATEGORIES.items()
    }


def scan_eval_file(path: str) -> dict:
    """Scan an eval file for safety refusal patterns.
    
    Returns dict with summary statistics and sample-level details.
    """
    result = {
        'path': path,
        'filename': os.path.basename(path),
        'total_samples': 0,
        'scored_samples': 0,
        'refusal_count': 0,
        'refusal_rate': 0.0,
        'mean_score': None,
        'non_zero_scores': 0,
        'category_counts': Counter(),
        'model': None,
        'status': None,
        'examples': [],
    }
    
    try:
        with zipfile.ZipFile(path) as z:
            # Read header for model info
            if 'header.json' in z.namelist():
                header = json.loads(z.read('header.json'))
                result['model'] = header.get('eval', {}).get('model', None)
                result['status'] = header.get('status', None)
            
            sample_files = sorted([
                n for n in z.namelist()
                if n.startswith('samples/') and n.endswith('.json')
            ])
            result['total_samples'] = len(sample_files)
            
            scores = []
            for sf in sample_files:
                raw = json.loads(z.read(sf))
                score_data = raw.get('scores', {}).get('saber_overall', {})
                score = score_data.get('value', None)
                answer = score_data.get('answer', '')
                task_id = raw.get('id', '')
                
                if score is not None:
                    result['scored_samples'] += 1
                    scores.append(score)
                    if score > 0:
                        result['non_zero_scores'] += 1
                
                if is_safety_refusal(answer):
                    result['refusal_count'] += 1
                    cats = classify_refusal(answer)
                    for cat, matched in cats.items():
                        if matched:
                            result['category_counts'][cat] += 1
                    
                    # Save first few examples
                    if len(result['examples']) < 3:
                        result['examples'].append({
                            'task_id': task_id,
                            'score': score,
                            'answer_preview': answer[:400],
                        })
            
            if scores:
                result['mean_score'] = sum(scores) / len(scores)
            if result['scored_samples'] > 0:
                result['refusal_rate'] = result['refusal_count'] / result['scored_samples']
    
    except Exception as e:
        result['error'] = str(e)
    
    return result


def find_eval_files(domain: str) -> list[str]:
    """Find all .eval files for a domain across configured directories."""
    paths = []
    seen_names = set()
    for d in EVAL_DIRS.get(domain, []):
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.endswith('.eval') and f not in seen_names:
                paths.append(os.path.join(d, f))
                seen_names.add(f)
    return paths


print('Helper functions loaded: is_safety_refusal, classify_refusal, scan_eval_file, find_eval_files')


---

## 1. Case Study: Claude Haiku 4.5 + GH Copilot (Excytin)

### Key Finding

Claude Haiku 4.5 paired with the GH Copilot agent architecture **refused to perform 100% of Excytin tasks**, citing safety concerns. The model interpreted the cybersecurity incident investigation prompts as potential social engineering and declined to:

1. **Connect to databases** — viewed credential usage as unsafe
2. **Execute SQL queries** — considered it unauthorized system access
3. **Analyze security data** — flagged the entire benchmark framing as suspicious

### Critical insight: Same model, different outcome

The **same model (Haiku 4.5) works fine with the React agent** — the refusal is specific to the Copilot agent architecture's prompt framing. This proves the refusal is **not a fundamental model limitation** but a **prompt-sensitivity issue** in how the agent presents the task.

### Implications

- Agent architecture comparison is invalidated when one configuration triggers 100% refusals
- Copilot's system prompt or tool presentation format triggers Haiku's safety guardrails differently than React's
- This pattern may affect other models with strict safety filters


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 1: Claude Haiku 4.5 + GH Copilot (Excytin) — Deep Dive
# ══════════════════════════════════════════════════════════════════════════════

# Find the haiku copilot eval file
haiku_copilot_path = None
for d in EVAL_DIRS.get('excytin', []):
    candidate = os.path.join(d, 'haiku_4.5_copilot.eval')
    if os.path.exists(candidate):
        haiku_copilot_path = candidate
        break

if haiku_copilot_path:
    haiku_result = scan_eval_file(haiku_copilot_path)
    
    print(f'╔══════════════════════════════════════════════════════════════════╗')
    print(f'║  CASE STUDY: Claude Haiku 4.5 (GH Copilot) — Excytin          ║')
    print(f'╠══════════════════════════════════════════════════════════════════╣')
    print(f'║  Model:                  {str(haiku_result["model"]):>38s} ║')
    print(f'║  Eval status:            {str(haiku_result["status"]):>38s} ║')
    print(f'║  Total samples in eval:  {haiku_result["total_samples"]:>5d}                                ║')
    print(f'║  Scored samples:         {haiku_result["scored_samples"]:>5d}                                ║')
    print(f'║  Safety refusals:        {haiku_result["refusal_count"]:>5d}  ({haiku_result["refusal_rate"]*100:.1f}%)                      ║')
    print(f'║  Non-zero scores:        {haiku_result["non_zero_scores"]:>5d}                                ║')
    print(f'║  Mean score:             {haiku_result["mean_score"] or 0:.4f}                              ║')
    print(f'╚══════════════════════════════════════════════════════════════════╝')
    print()
    
    # Refusal category breakdown
    print('Refusal Categories (overlapping — a single refusal cites multiple concerns):')
    scored = max(haiku_result['scored_samples'], 1)
    for cat, count in haiku_result['category_counts'].most_common():
        print(f'  {cat:40s}  {count:>4d}/{scored}  ({count/scored*100:.1f}%)')
    
    # ── Visualization ──
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Panel 1: Refusal category bar chart
    ax = axes[0]
    cats = [c for c, _ in haiku_result['category_counts'].most_common()]
    vals = [haiku_result['category_counts'][c] / scored * 100 for c in cats]
    if cats:
        bars = ax.barh(range(len(cats)), vals, color='#E74C3C', alpha=0.8,
                       edgecolor='black', linewidth=0.5)
        ax.set_yticks(range(len(cats)))
        ax.set_yticklabels(cats, fontsize=9)
        ax.set_xlabel('% of Scored Samples')
        ax.set_title('Safety Refusal Categories\n(Haiku 4.5 + GH Copilot)', fontweight='bold')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%', va='center', fontweight='bold', fontsize=10)
        ax.set_xlim(0, 110)
    
    # Panel 2: Compare with Haiku React
    ax = axes[1]
    haiku_react_path = None
    for d in EVAL_DIRS.get('excytin', []):
        candidate = os.path.join(d, 'haiku_4.5_react.eval')
        if os.path.exists(candidate):
            haiku_react_path = candidate
            break
    
    if haiku_react_path:
        react_result = scan_eval_file(haiku_react_path)
        react_scores = []
        with zipfile.ZipFile(haiku_react_path) as z:
            for sf in sorted(n for n in z.namelist() if n.startswith('samples/') and n.endswith('.json')):
                raw = json.loads(z.read(sf))
                s = raw.get('scores', {}).get('saber_overall', {}).get('value', None)
                if s is not None:
                    react_scores.append(s)
        
        if react_scores:
            ax.hist(react_scores, bins=20, color='#2CA02C', alpha=0.7,
                    edgecolor='black', linewidth=0.5,
                    label=f'Haiku 4.5 (React)\nmean={np.mean(react_scores):.3f}')
            ax.axvline(0.0, color='#E74C3C', linewidth=3, linestyle='--',
                       label=f'Haiku 4.5 (GH Copilot)\nALL samples = 0.0 (refused)')
            ax.set_xlabel('Score')
            ax.set_ylabel('Count')
            ax.set_title('Same Model, Different Agent Architecture\n'
                         'React works — Copilot triggers safety refusal', fontweight='bold')
            ax.legend(fontsize=9, loc='upper right')
        else:
            ax.text(0.5, 0.5, 'No React scores available', ha='center', va='center',
                    transform=ax.transAxes, fontsize=12)
    else:
        ax.text(0.5, 0.5, 'Haiku 4.5 React eval not found', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
    
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / 'haiku_copilot_safety_refusal.png', bbox_inches='tight')
    plt.show()
    
    # Example refusal
    if haiku_result['examples']:
        print()
        print('── Example Refusal (truncated) ──')
        ex = haiku_result['examples'][0]
        print(f'Task: {ex["task_id"]}')
        print(f'Score: {ex["score"]}')
        for line in textwrap.wrap(ex['answer_preview'], width=100):
            print(f'  {line}')
else:
    print('⚠ haiku_4.5_copilot.eval not found in any configured directory')
    print('  Place it in eval_samples/excytin/ or latest_experiments/excytin/')


---

## 2. Cross-Model Safety Refusal Scan

### What this measures

Systematically scans **all eval files** across all domains to detect safety refusal patterns. This identifies:
- Which model + agent combinations have elevated refusal rates
- Whether refusals are isolated incidents or systemic patterns
- Domain-specific vs. model-specific refusal triggers

### How to read the output

- Files with >50% refusal rate are flagged as critical (red)
- Files with 5–50% are flagged as warnings (amber)
- Files with <5% are informational (blue)
- A file with 0 refusals means all samples engaged with the benchmark task

### Key question

Is the Haiku + Copilot refusal an isolated incident, or are there other model × agent × domain combinations where safety filters interfere with benchmark execution?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANALYSIS 2: Cross-Model Safety Refusal Scan
# ══════════════════════════════════════════════════════════════════════════════

all_results = []

for domain in EVAL_DIRS:
    eval_files = find_eval_files(domain)
    print(f'Scanning {domain}: {len(eval_files)} eval files')
    
    for path in eval_files:
        result = scan_eval_file(path)
        result['domain'] = domain
        all_results.append(result)
        
        # Report files with non-trivial refusal rates
        if result['refusal_count'] > 0 and result['scored_samples'] > 0:
            rate = result['refusal_rate'] * 100
            symbol = '🚨' if rate > 50 else '⚠️' if rate > 5 else 'ℹ️'
            print(f'  {symbol} {result["filename"]:50s} '
                  f'refusals={result["refusal_count"]}/{result["scored_samples"]} '
                  f'({rate:.1f}%)  model={result["model"]}')

print()

# Build summary DataFrame
scan_df = pd.DataFrame([{
    'domain': r['domain'],
    'filename': r['filename'],
    'model': r['model'],
    'status': r['status'],
    'total_samples': r['total_samples'],
    'scored_samples': r['scored_samples'],
    'refusal_count': r['refusal_count'],
    'refusal_rate': r['refusal_rate'],
    'non_zero_scores': r['non_zero_scores'],
    'mean_score': r['mean_score'],
} for r in all_results if r['scored_samples'] > 0])

if len(scan_df) > 0:
    # Show files with any refusals
    refusal_files = scan_df[scan_df['refusal_count'] > 0].sort_values('refusal_rate', ascending=False)
    
    print(f'\nSafety Refusal Summary: {len(refusal_files)} of {len(scan_df)} eval files have refusals')
    print()
    
    if len(refusal_files) > 0:
        print(f'{"Domain":>10s} {"File":50s} {"Model":35s} {"Refusals":>10s} {"Rate":>8s} {"Mean Score":>12s}')
        print('-' * 130)
        for _, row in refusal_files.iterrows():
            print(f'{row["domain"]:>10s} {row["filename"]:50s} '
                  f'{str(row["model"]):35s} '
                  f'{row["refusal_count"]:>4d}/{row["scored_samples"]:<5d} '
                  f'{row["refusal_rate"]*100:>6.1f}% '
                  f'{row["mean_score"]:>10.4f}')
    else:
        print('No safety refusals detected in any eval file.')
else:
    print('No scored eval files found.')


### Refusal Rate Visualization

### What this shows

Two charts comparing refusal rates across all model × agent configurations:

1. **Left panel: Refusal rate by eval file** — horizontal bar chart showing which configurations have the highest refusal rates, color-coded by severity (red = critical >50%, amber = warning 5–50%, blue = low <5%)
2. **Right panel: Refusal rate vs. mean score scatter** — shows the inverse relationship between refusal rate and performance. Configurations with high refusal rates cluster near the bottom (low scores), while those with no refusals spread across the score range

### Key insight

This visualization answers: "How widespread is the safety refusal problem?" If the pattern is isolated to one configuration, it's a niche issue. If multiple configurations show elevated rates, it indicates a systemic challenge for cybersecurity benchmarking.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION: Refusal Rates Across Configurations
# ══════════════════════════════════════════════════════════════════════════════

if len(scan_df) > 0 and scan_df['refusal_count'].sum() > 0:
    refusal_files = scan_df[scan_df['refusal_count'] > 0].sort_values('refusal_rate', ascending=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(18, max(5, len(refusal_files) * 0.5)))
    
    # Panel 1: Refusal rate by eval file
    ax = axes[0]
    labels = [f'{row["domain"]}/{row["filename"][:35]}' for _, row in refusal_files.iterrows()]
    rates = refusal_files['refusal_rate'].values * 100
    colors = ['#E74C3C' if r > 50 else '#F39C12' if r > 5 else '#3498DB' for r in rates]
    
    bars = ax.barh(range(len(labels)), rates, color=colors, alpha=0.8,
                   edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Safety Refusal Rate (%)')
    ax.set_title('Safety Refusal Rate by Eval File', fontweight='bold')
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{rate:.1f}%', va='center', fontweight='bold', fontsize=9)
    ax.set_xlim(0, max(rates) * 1.15 if len(rates) > 0 else 100)
    
    # Legend
    legend_patches = [
        mpatches.Patch(color='#E74C3C', alpha=0.8, label='Critical (>50%)'),
        mpatches.Patch(color='#F39C12', alpha=0.8, label='Warning (5-50%)'),
        mpatches.Patch(color='#3498DB', alpha=0.8, label='Low (<5%)'),
    ]
    ax.legend(handles=legend_patches, loc='lower right', fontsize=8)
    
    # Panel 2: Refusal rate vs mean score scatter
    ax = axes[1]
    for _, row in scan_df.iterrows():
        color = '#E74C3C' if row['refusal_rate'] > 0.5 else '#F39C12' if row['refusal_rate'] > 0.05 else '#2CA02C'
        alpha = 0.9 if row['refusal_count'] > 0 else 0.3
        ax.scatter(row['refusal_rate'] * 100, row['mean_score'] or 0,
                   color=color, s=80, alpha=alpha, edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Safety Refusal Rate (%)')
    ax.set_ylabel('Mean Score')
    ax.set_title('Refusal Rate vs. Performance', fontweight='bold')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / 'cross_model_refusal_scan.png', bbox_inches='tight')
    plt.show()
else:
    print('No refusals detected — nothing to visualize.')
    print('This is a good sign! All models engaged with the benchmark tasks.')


---

## Next Steps

### Extending this analysis

- **New model evals:** Re-run this notebook after adding new model evaluations to detect emerging refusal patterns
- **Prompt engineering:** For models that refuse, investigate which prompt components trigger the refusal and test mitigations (e.g., adding explicit authorization context, modifying the system prompt framing)
- **Agent architecture comparison:** Cross-reference with [`agent_architecture_analysis.ipynb`](agent_architecture_analysis.ipynb) to understand how agent design affects refusal rates
- **Per-task analysis:** Some tasks may be more prone to triggering refusals than others — analyze task-level refusal distributions to identify which task attributes (keywords, scenario descriptions) are most sensitive

### Implications for benchmark design

- Safety refusals produce **0.0 scores** that conflate two very different phenomena: "model tried and failed" vs. "model refused to try"
- Future scoring could separate these outcomes with a distinct `refused` status
- Agent architectures should be evaluated for their "safety compatibility" with each model before running full benchmarks
- Benchmark prompts may need domain-specific safety disclaimers (e.g., "This is an authorized security evaluation in a sandboxed environment") to avoid false-positive refusals